# Import Statements

In [63]:
import pandas as pd
df = pd.read_csv("2026dependenciesHWFile.csv")
left_df = df.iloc[:6091,:7]
right_df = df.iloc[:6091,8:].dropna()
right_df.rename(columns={'courseLetter.1': 'courseLetter', 'courseNumber.1': 'courseNumber'}, inplace=True)
# right_df
# left_df

### Computing Courses

**Question 1**\
How many CIS courses at the 100 level are part of the Computer Information Systems, BS program?

In [64]:
left_df['courseNumber'] = left_df['courseNumber'].apply(str)
cis100_bs_df = left_df[
    (left_df['courseLetter'] == 'CIS') &
    (left_df['relatedTo'] == 'Computer Information Systems, BS') &
    (left_df['courseNumber'].str.startswith('1'))
    ]
cis100_bs_df.shape[0]

5

**Question 2**\
How many courses are offered with the CIS designation?

In [65]:
cis_total = (right_df['courseLetter'] == 'CIS').sum()
print(cis_total)

76


**Question 3**\
How many courses are offered with the CYB designation?

In [66]:
cyb_total = (right_df['courseLetter'] == 'CYB').sum()
print(cyb_total)

40


**Question 4**\
How many courses are offered with the DSC designation?

In [67]:
dsc_total = (right_df['courseLetter'] == 'DSC').sum()
print(dsc_total)

23


**Question 5**\
How many courses are offered with the ISS designation?

In [68]:
iss_total = (right_df['courseLetter'] == 'ISS').sum()
print(iss_total)

54


### Crosslisting of Courses

**Create a dataframe with course designated as "CrossList"**

In [69]:
crossList_df = left_df[left_df['How'] == 'CrossList']
# crossList_df.shape[0]

**Question 6**\
Find how many cross lists each course has.  How may courses are crosslisted with POS 223?

In [70]:
cl_each_course_df = pd.DataFrame(
    [i, int((crossList_df['relatedTo'] == i).sum())] for i in left_df['relatedTo']
    )
cl_each_course_df.columns = ['course_name', 'cl']
pos_223_cl_total = (crossList_df['relatedTo'] == 'POS 223').sum()
print(pos_223_cl_total)

2


**Question 7**\
What is the mean number of crosslists per course?

In [71]:
total_crosslists = (left_df['How'] == 'CrossList').sum()
course_dict_total = right_df.value_counts().sum()
# print(total_crosslists)
# print(course_dict_total)
mean_crosslist_course = total_crosslists / course_dict_total
print(f"{mean_crosslist_course:.2f}")

0.28


**Question 8**\
What is the median number of courses crosslisted?

In [72]:
median_cl_per_course = cl_each_course_df['cl'].median()
print(f"{median_cl_per_course:.2f}")

0.00


**Question 9**\
What is the mode of the number of crosslisted courses?

In [73]:
print(cl_each_course_df['cl'].mode())

0    0
Name: cl, dtype: int64


**Question 10**\
Given that a course is actually crosslisted, what is the average number of courses it is crosslisted to?

In [74]:
course_cl = crossList_df.groupby('course')['relatedTo'].count()
cl_mean = course_cl.mean()
print(f"{cl_mean:.2f}")

1.39


**Question 11**\
How many courses have four designations (crosslisted designations count as 1 course in total)?  Note:  what would that look like in this dataset?

In [75]:
courses_with_four_designations = course_cl.value_counts()[3] / 4
print(int(courses_with_four_designations))

5


**Question 12**\
Assuming all of the crosslists are one course (e.g. CIS 255 and DSC 255 are one course), how many courses does UMA offer in its catalog?

In [76]:
# print(course_cl.value_counts())
duplicates  = int((course_cl.value_counts()[1] / 2) + (course_cl.value_counts()[2] / 3) + (course_cl.value_counts()[3] / 4))
# print(duplicates)
total_cl_courses = course_cl.value_counts().sum()
# print(total_cl_courses)
course_dict_total = right_df.value_counts().sum()
# print(course_dict_total)
total_UMA_courses = course_dict_total - (total_cl_courses - duplicates)
print(total_UMA_courses)

1214


**Question 13**\
All four of the designations CIS, CYB, DSC, and ISS are offered by the computing group.  How many specific courses are offered by the computing group.  Note:  courses crosslisted together are ONE course for this purpose.

In [77]:
comp_total = right_df['courseLetter'].isin(['CIS', 'CYB', 'DSC', 'ISS']).sum()
# print(comp_total)
duplicates = (
    (left_df['How'] == 'CrossList') &
    (left_df['courseLetter'].isin(['CIS', 'CYB', 'DSC', 'ISS'])) &
    (left_df['relatedToCourseLetter'].isin(['CIS', 'CYB', 'DSC', 'ISS']))
    ).sum()
# print(duplicates)
computing_group_courses = comp_total - int((duplicates / 2))
print(computing_group_courses)

182


### Architecture Program
In this section, we will be studying the courses and programs offered by the Architecture faculty.

**Question 14**\
What proportion of courses (by code – crosslists are separate for this purpose) are in the Architecture, B.Arch checksheet?

In [78]:
arch_checksheet_total = (left_df['relatedTo'] == 'Architecture, B.Arch').sum()
proportion_barch = (arch_checksheet_total / course_dict_total)
# print(arch_checksheet_total)
# print(course_dict_total)
print(f"{proportion_barch:.2f}")

0.04


**Question 15**\
What proportion of courses (by code) are ARC courses?

In [79]:
arc_total = (right_df['courseLetter'] == 'ARC').sum()
proportion_arc = arc_total / course_dict_total
# print(arc_total)
# print(course_dict_total)
print(f"{proportion_arc:.2f}")

0.03


**Question 16**\
Assuming these are independent, what proportion of courses (by code) are both in the Architecture, B.Arch checksheet and are ARC courses?

In [80]:
barch_arc_total = ((left_df['relatedTo'] == 'Architecture, B.Arch') &
                   (left_df['courseLetter'] == 'ARC')).sum()
# print(barch_arc_total)
# print(arch_checksheet_total)
proportion_barch_arc = barch_arc_total / arch_checksheet_total
print(f"{proportion_barch_arc:.2f}")

0.60


**Question 17**\
What is the conditional probability that a given course with an ARC designation is part of the Architecture, B.Arch checksheet?

In [81]:
# To answer this, P(A|B) or the probability of A given that B has happened.
# A = total Architecture, B.Arch.
# B = total courses with ARC designation
# P(total Architecture, B.Arch. | total courses with ARC designation)

# print(barch_arc_total)
# print(arc_total)
cp_barch_arc = barch_arc_total / arc_total
print(f"{cp_barch_arc:.2f}")

0.76


**Question 18**\
What is the conditional probability that a given course in the Architecture, B.Arch checksheet is an ARC course?

In [82]:
# To answer this, P(A|B) or the probability of A given that B has happened.
# A = total courses with ARC designation
# B = total Architecture, B.Arch.
# P(total courses with ARC designation. | total Architecture, B.Arch)

# print(barch_arc_total)
# print(arch_checksheet_total)
cp_barch_arc = barch_arc_total / arch_checksheet_total
print(f"{cp_barch_arc:.2f}")

0.60


**Question 19**\
Do the answers from 14-18 suggest that ARC courses and Architecture, B.Arch checksheet courses are independent statistically?  Why or why not.

### Prerequisites
In this section, we will address whether or not there is a bidirectional relationship over prerequisites (namely are courses equally likely to have prerequisites or to be prerequisites).

**Question 20**\
What proportion of courses (by code) are a prerequisite for another course?

In [83]:
prereq_course_df = left_df[left_df['How'] == 'Prereq'][['course', 'How']]
prereq_course_total = prereq_course_df['course'].nunique()
# print(prereq_course_total)
# print(course_dict_total)
proportion_course_prereq = prereq_course_total / course_dict_total
print(f"{proportion_course_prereq:.2f}")

0.27


**Question 21**\
What proportion of courses (by code) have at least one prerequisite?

In [84]:
prereq_relatedTo_df = left_df[left_df['How'] == 'Prereq'][['relatedTo', 'How']]
prereq_relatedTo_total = prereq_relatedTo_df['relatedTo'].nunique()
# print(prereq_relatedTo_total)
# print(course_dict_total)
proportion_relatedTo_prereq = prereq_relatedTo_total / course_dict_total
print(f"{proportion_relatedTo_prereq:.2f}")


0.65


### American Studies
The Curriculum Committee has had a history of challenges with the American Studies program.  The coordinator of the program has what seems to be a liberal crosslisting policy.  This makes things difficult for those who maintain MaineStreet and maintain Acalog.  You will be addressing whether or not the data supports the claim of the Curriculum Committee that the coordinator of the program doth crosslist too much vis a vis their peers.

Recall:
An association rule has an antecedent and a consequent.  The antecedent begins the rule, while the consequent ends the rule.  If antecedent, then consequent.

**Support** is the proportion an item occurs in the dataset\
**Confidence** is the proportion of the time that given the antecedent, the consequent occurs.\
**Lift** is the ratio between the confidence of the rule and the support of its consequent.

**Question 22**\
What is the support for a course being an AME course?

In [85]:
ame_total = (right_df['courseLetter'] == 'AME').sum()
# print(ame_total)
# print(course_dict_total)
ame_support = ame_total / course_dict_total
print(f"{ame_support:.2f}")

0.02


**Question 23**\
What is the support for a course being crosslisted?

In [86]:
# total_cl = (left_df['How'] == 'CrossList').sum()
# print(total_cl_courses)
# print(course_dict_total)
crossList_support = total_cl_courses / course_dict_total
print(f"{crossList_support:.2f}")

0.20


**Question 24**\
What is the confidence that an AME course is crosslisted?

In [87]:
# Confidence(AME -> Crosslisted) = AME / AME corsslisted)
crosslist_df = left_df[left_df['How'] == 'CrossList']
concat_courses = pd.concat([
    crosslist_df['course'],
    crosslist_df['relatedTo']
    ], ignore_index=True)
ame_cl_df = pd.DataFrame({'crosslisted_courses': concat_courses})
ame_cl_df = ame_cl_df.drop_duplicates(ignore_index=True)
ame_cl = ame_cl_df['crosslisted_courses'].astype(str).str.contains('AME', case=False, na=False)
# print(ame_cl.sum())
# print(ame_total)
ame_confidence = ame_cl.sum() / ame_total
print(f"{ame_confidence:.2f}")


0.81


**Question 25**\
What is the lift associated with knowing that a course is an AME course?

In [88]:
# Lift(AME confidence -> ame support Crosslisted) = AME confidence / ame support Crosslisted
ame_lift = ame_confidence / crossList_support
print(f"{ame_lift:.2f}")

4.12
